# 03. Ablation Study

Study of the influence of components: BatchNorm, Dropout, Focal Loss.

In [ ]:
import torch
import sys
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from torchvision.ops import sigmoid_focal_loss

sys.path.append('..')
from rs.model import PositionPredictor, FocalLoss
from rs.training import train_model
from rs.evaluation import evaluate_fsr
from rs.channels import qsc_erasure_channel
from rs.dataset_gen import RSPositionDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
P_ERR, P_ERASE = 0.02, 0.06
TRAIN_SIZE, EPOCHS = 50000, 150

print('Generating dataset...')
dataset = RSPositionDataset(TRAIN_SIZE, P_ERR, P_ERASE)
print(f'Generated {len(dataset)} examples.')

Generating dataset...
Generated 50000 examples.


In [9]:
class ConfigurableModel(nn.Module):
    def __init__(self, use_bn=True, dropout=0.1):
        super().__init__()
        layers = []
        in_size = 511
        for _ in range(4):
            layers.append(nn.Linear(in_size, 512))
            if use_bn: layers.append(nn.BatchNorm1d(512))
            layers.append(nn.ReLU())
            if dropout > 0: layers.append(nn.Dropout(dropout))
            in_size = 512
        layers.append(nn.Linear(512, 255))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x): return self.net(x)

def run_experiment(name, use_bn, dropout, loss_type, alpha=0.25, gamma=1.0):
    print(f'{name}...', end=' ')
    model = ConfigurableModel(use_bn=use_bn, dropout=dropout).to(device)
    
    if loss_type == 'BCE':
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = lambda i, t: sigmoid_focal_loss(i, t, alpha=alpha, gamma=gamma, reduction='mean')
    
    train_model(model, dataset, criterion, epochs=EPOCHS, device=device, verbose=False)
    fsr = evaluate_fsr(model, qsc_erasure_channel, P_ERR, P_ERASE, device=device)
    print(f'FSR = {fsr:.1%}')
    return fsr, model

In [10]:
sequential_results = []

# Baseline: BCE w/o regularization
fsr, _ = run_experiment('BCE (baseline)', use_bn=False, dropout=0.0, loss_type='BCE')
sequential_results.append({'config': 'BCE (baseline)', 'fsr': fsr})

# BCE + BN
fsr, _ = run_experiment('BCE + BN', use_bn=True, dropout=0.0, loss_type='BCE')
sequential_results.append({'config': 'BCE + BN', 'fsr': fsr})

# BCE + Dropout
fsr, _ = run_experiment('BCE + Dropout', use_bn=False, dropout=0.1, loss_type='BCE')
sequential_results.append({'config': 'BCE + Dropout', 'fsr': fsr})

# BCE + Dropout + BN
fsr, _ = run_experiment('BCE + Dropout', use_bn=True, dropout=0.1, loss_type='BCE')
sequential_results.append({'config': 'BCE + Dropout', 'fsr': fsr})

# Only Focal Loss
fsr, _ = run_experiment('Focal', use_bn=False, dropout=0.0, loss_type='Focal')
sequential_results.append({'config': 'Focal', 'fsr': fsr})

# Focal Loss + BN
fsr, _ = run_experiment('Focal + BN', use_bn=True, dropout=0.0, loss_type='Focal')
sequential_results.append({'config': 'Focal + BN', 'fsr': fsr})

# Focal Loss + Dropout
fsr, _ = run_experiment('Focal + Dropout', use_bn=False, dropout=0.1, loss_type='Focal')
sequential_results.append({'config': 'Focal + Dropout', 'fsr': fsr})

# Focal Loss + BN + Dropout
fsr, _ = run_experiment('Focal + Dropout', use_bn=True, dropout=0.1, loss_type='Focal')
sequential_results.append({'config': 'Focal + Dropout', 'fsr': fsr})

BCE (baseline)... FSR = 66.2%
BCE + BN... FSR = 42.8%
BCE + Dropout... FSR = 69.8%
BCE + Dropout... FSR = 83.3%
Focal... FSR = 71.8%
Focal + BN... FSR = 48.3%
Focal + Dropout... FSR = 73.8%
Focal + Dropout... FSR = 81.4%


In [11]:
df_seq = pd.DataFrame(sequential_results)
print('Последовательное добавление компонентов:')
print(df_seq.to_string(index=False))

Последовательное добавление компонентов:
         config   fsr
 BCE (baseline) 0.662
       BCE + BN 0.428
  BCE + Dropout 0.698
  BCE + Dropout 0.833
          Focal 0.718
     Focal + BN 0.483
Focal + Dropout 0.738
Focal + Dropout 0.814


In [12]:
focal_params = [
    (0.15, 1.5), (0.15, 2.0), 
    (0.20, 1.5), (0.20, 2.0), 
    (0.25, 1.0), (0.30, 1.0)
]

focal_results = []
for alpha, gamma in focal_params:
    name = f'α={alpha}, γ={gamma}'
    fsr, _ = run_experiment(name, use_bn=True, dropout=0.1, loss_type='Focal', alpha=alpha, gamma=gamma)
    focal_results.append({'alpha': alpha, 'gamma': gamma, 'fsr': fsr})

α=0.15, γ=1.5... FSR = 82.9%
α=0.15, γ=2.0... FSR = 82.5%
α=0.2, γ=1.5... FSR = 84.2%
α=0.2, γ=2.0... FSR = 82.7%
α=0.25, γ=1.0... FSR = 83.4%
α=0.3, γ=1.0... FSR = 80.8%


In [14]:
df_focal = pd.DataFrame(focal_results)
print('Focal Loss tuning:')
print(df_focal.to_string(index=False))

best_focal = df_focal.loc[df_focal['fsr'].idxmax()]
print(f"\nBest parameters: α={best_focal['alpha']}, γ={best_focal['gamma']}, FSR={best_focal['fsr']:.1%}")

Focal Loss tuning:
 alpha  gamma   fsr
  0.15    1.5 0.829
  0.15    2.0 0.825
  0.20    1.5 0.842
  0.20    2.0 0.827
  0.25    1.0 0.834
  0.30    1.0 0.808

Best parameters: α=0.2, γ=1.5, FSR=84.2%


In [15]:
print('Final model training...')
final_model = ConfigurableModel(use_bn=True, dropout=0.1).to(device)
criterion = lambda i, t: sigmoid_focal_loss(i, t, alpha=0.2, gamma=1.5, reduction='mean')
train_model(final_model, dataset, criterion, epochs=EPOCHS, device=device, verbose=False)
print('Training finished.')

Final model training...
Training finished.
